In [1]:
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.stats import chi2_contingency
from itertools import combinations
from plotly.subplots import make_subplots


In [2]:
df = pd.read_csv("cleaned_data.csv")

In [3]:
df.describe()


,Age,Sleep Hours,Physical Activity (hrs/week),Caffeine Intake (mg/day),Alcohol Consumption (drinks/week),Stress Level (1-10),Heart Rate (bpm),Breathing Rate (breaths/min),Sweating Level (1-5),Therapy Sessions (per month),...,Physical Activity (hrs/week)_missing,Caffeine Intake (mg/day)_missing,Alcohol Consumption (drinks/week)_missing,Heart Rate (bpm)_missing,Breathing Rate (breaths/min)_missing,Therapy Sessions (per month)_missing,Stress Level (1-10)_missing,Sweating Level (1-5)_missing,Diet Quality (1-10)_missing,Anxiety Level (1-10)_missing
count,2030.000000,2030.000000,2030.000000,2030.000000,2030.000000,2030.000000,2030.000000,2030.000000,2030.000000,2030.000000,...,2030.000000,2030.000000,2030.000000,2030.0,2030.0,2030.0,2030.0,2030.0,2030.000000,2030.0
mean,39.921748,6.671750,2.975829,294.047650,9.786432,6.028571,92.083744,20.955665,3.095567,2.250739,...,0.019704,0.046305,0.019704,0.0,0.0,0.0,0.0,0.0,0.038424,0.0
std,13.039840,1.189402,1.778277,149.910903,5.629003,3.137269,18.929407,5.182524,1.392697,1.890728,...,0.139017,0.210198,0.139017,0.0,0.0,0.0,0.0,0.0,0.192264,0.0
min,18.000000,3.500000,0.000000,0.000000,0.000000,1.000000,60.000000,12.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
25%,29.000000,6.000000,1.600000,183.000000,5.000000,3.000000,76.000000,17.000000,2.000000,1.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
50%,39.921748,6.700000,2.850000,284.000000,10.000000,6.000000,93.000000,21.000000,3.000000,2.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
75%,50.000000,7.500000,4.200000,384.000000,15.000000,9.000000,107.000000,26.000000,4.000000,3.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
max,64.000000,9.900000,8.250000,710.875000,19.000000,15.000000,153.500000,29.000000,5.000000,6.000000,...,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,1.000000,0.0


In [4]:
df.head()

,Age,Gender,Occupation,Sleep Hours,Physical Activity (hrs/week),Caffeine Intake (mg/day),Alcohol Consumption (drinks/week),Smoking,Family History of Anxiety,Stress Level (1-10),...,Binned_Caffeine Intake (mg/day),Binned_Alcohol Consumption (drinks/week),Binned_Heart Rate (bpm),Binned_Breathing Rate (breaths/min),Binned_Therapy Sessions (per month),Binned_Stress Level (1-10),Binned_Diet Quality (1-10),Binned_Anxiety Level (1-10),Occupation_Group,Therapy_Group
0,59.0,Other,Teacher,7.0,2.4,40.0,5.0,Yes,No,4.0,...,Low,Light,Normal,Rapid,No Therapy,Moderate Stress,Unhealthy,Low Anxiety,Academic,No Therapy
1,46.0,Female,Student,5.1,5.4,156.0,11.0,Yes,No,3.0,...,Moderate,Heavy,Normal,Normal,Occasional,Low Stress,Unhealthy,Moderate Anxiety,Academic,Therapy
2,40.0,Other,Lawyer,5.1,1.9,570.0,14.0,Yes,Yes,9.0,...,High,Heavy,High,Rapid,Intensive,High Stress,Average,High Anxiety,Professional,Therapy
3,40.0,Male,Nurse,7.6,0.9,129.0,0.0,No,No,9.0,...,Moderate,NaN,High,Normal,Occasional,High Stress,Unhealthy,Moderate Anxiety,Healthcare,Therapy
4,26.0,Male,Other,6.7,3.0,64.0,13.0,No,No,15.0,...,Low,Heavy,High,Normal,No Therapy,NaN,Average,Low Anxiety,Other,No Therapy


In [5]:
def shapiro_test(data, cl):
    a = data[cl]
    shapiro_stat, shapiro_p = stats.shapiro(a)
    
    print(f"Normality Test")
    print("-" * 20)
    print(f"Shapiro-Wilk: Statistic = {shapiro_stat:.4f}, p-value = {shapiro_p:.4f}") 
    
    
    alpha = 0.05
    if shapiro_p > alpha:
        print("fail to reject h0")
    else:
        print("reject h0")


In [6]:
def one_sample_ttest(data, cl, mu0):
    a = data[cl]
    n = len(a)
    mean = np.mean(a)
    s = np.std(a, ddof=1)
    se = s / np.sqrt(n)
    t_stat = (mean - mu0) / se
    df = n - 1
    p_valeu = 2 * (1 - stats.t.cdf(abs(t_stat), df))

    t_critical = stats.t.ppf(0.975, df)
    margin = t_critical * se
    ci_lower = mean - margin
    ci_upper = mean + margin


    print(f"One-Sample t-Test")
    print("-" * 20)
    print(f"n = {n}")
    print(f"x̄ = {mean:.3f}")
    print(f"se = {se:.4f}")
    print(f"t = {t_stat:.4f}")
    print(f"p-value = {p_valeu:.4f}")
    print(f"CI 95% = [{ci_lower:.3f}, {ci_upper:.3f}]")
    
    if p_valeu < 0.05:
        print("reject h0")
    else:
        print("fail to reject h0")


In [7]:

def two_sample_ttest(df, cl_num, cl_target, data1, data2):

    group1 = df[df[cl_target] == data1][cl_num]
    group2 = df[df[cl_target] == data2][cl_num]
    
    n1, n2 = len(group1), len(group2)
    mean1, mean2 = np.mean(group1), np.mean(group2)
    var_A = ((group1 - mean1)** 2).sum() / (len(group1)-1)
    var_B = ((group2 - mean2)** 2).sum() / (len(group2)-1)
    sp = np.sqrt(((n1 - 1 )* var_A + (n2 - 1) * var_B)/ (n1 + n2 - 2))
    
    t_stat = (mean1 - mean2) / (sp * np.sqrt(1/n1 + 1/n2))

    df = n1 + n2 - 2
    
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat),df))

    t_critical = stats.t.ppf(0.975, df)
    margin = t_critical * sp * np.sqrt(1/n1 + 1/n2)
    ci_lower = (mean1 - mean2) - margin
    ci_upper = (mean1 - mean2) + margin


    print(f"two_sample_ttest")
    print("-" * 20)
    print(f"1mean: {mean1:.3f}")
    print(f"2mean: {mean2:.3f}")
    print(f"t = {t_stat:.4f}")
    print(f"p-value = {p_value:.4f}")
    
    if p_value < 0.05:
        print("reject h0")
    else:
        print("fail to reject h0")

In [8]:
def chi_2_test(data, cl1, cl2):

    contingency_table = pd.crosstab(data[cl1], data[cl2])
    
    chi2_stat, p_value, dof, expected_table = stats.chi2_contingency(contingency_table)
    
    print(f"Chi-2 Test")
    print("-" * 20)
    print(f"Chi-2= {chi2_stat:.4f}")
    print(f"p-value = {p_value:.4f}")
    print(f"df = {dof}")
    
    if p_value < 0.05:
        print("reject h0")
    else:
        print("fail to reject h0(independent)")


In [9]:
def mann_whitney(df, cl_num, cl_target, data1, data2, alternative='two-sided'):
    group1 = df[df[cl_target] == data1][cl_num].dropna()
    group2 = df[df[cl_target] == data2][cl_num].dropna()
    u_stat, p_value = stats.mannwhitneyu(group1, group2, alternative=alternative)
    
    print(f"U statistic = {u_stat:.4f}")
    print(f"p-value = {p_value:.4f}")

In [10]:
def compare(df, cl_num, cl_target, data1, data2):
    group1 = df[df[cl_target] == data1][cl_num]
    group2 = df[df[cl_target] == data2][cl_num]

    t1, p1 = stats.shapiro(group1)
    t2, p2 = stats.shapiro(group2)

    print(f"normality test: {p1:.4f},  {p2:.4f}")

    if p1> 0.05 and p2>0.05 :
        print("use ttest")
        print("-"*20)
        two_sample_ttest(df, cl_num, cl_target, data1, data2)

    else:
        print("use mann_whitney")
        mann_whitney(df, cl_num, cl_target, data1, data2)

In [11]:

def correlation(data, cl1, cl2):
    group1 = data[cl1]
    group2 = data[cl2]

    t1, p1 = stats.shapiro(group1)
    t2, p2 = stats.shapiro(group2)

    print(f"normality test: {p1:.4f},  {p2:.4f}")

    if p1> 0.05 and p2>0.05 :
        corr_value, p_value = stats.pearsonr(group1, group2)
        name = 'pearson'

    else:
        corr_value, p_value = stats.spearmanr(group1, group2)
        name = 'spearman'

    print(f"{name}")
    print(f"correlation : {corr_value:.4f}")
    print(f"p_value: {p_value:.4f}")

The analysis compares the sample’s average sleep duration
 to the standard recommendation of 7 hours. With a p-value of 0.0000, the difference is statistically significant. This indicates that the study population, on average, sleeps significantly less than the recommended 7 hours per night

In [12]:
one_sample_ttest(df, "Sleep Hours" , 7)

One-Sample t-Test
--------------------
n = 2030
x̄ = 6.672
se = 0.0264
t = -12.4344
p-value = 0.0000
CI 95% = [6.620, 6.724]
reject h0


This analysis compares the sample’s average caffeine intake to the reference value of 250 mg/day. The result is highly significant (p = 0.0000). This indicates that the population’s caffeine consumption is statistically significantly higher than the comparison baseline of 250 mg/day

In [13]:
one_sample_ttest(df, 'Caffeine Intake (mg/day)', 250 )

One-Sample t-Test
--------------------
n = 2030
x̄ = 294.048
se = 3.3272
t = 13.2385
p-value = 0.0000
CI 95% = [287.522, 300.573]
reject h0


The analysis compares the sample’s mean heart rate
to the global standard average of 72 bpm. Since the p-value is 0.0000, the result is highly significant. This indicates that the heart rate in this population is statistically and significantly higher than the global norm.

In [14]:
one_sample_ttest(df, 'Heart Rate (bpm)', 72)

One-Sample t-Test
--------------------
n = 2030
x̄ = 92.084
se = 0.4201
t = 47.8031
p-value = 0.0000
CI 95% = [91.260, 92.908]
reject h0


Does the presence of a family history of anxiety significantly influence perceived stress levels in individuals?

In [15]:
compare(df, 'Stress Level (1-10)', 'Family History of Anxiety', 'Yes', 'No')

normality test: 0.0000,  0.0000
use mann_whitney
U statistic = 539585.5000
p-value = 0.0530


The p-value of 0.0530 is very close to the standard significance threshold (0.05). Strictly speaking, it is not statistically significant (p > 0.05). However, as a “borderline” result, it suggests a potential trend that might become significant with a slightly larger sample size or further data cleaning. It should be reported cautiously as a “marginal association.

In [16]:
mann_whitney(df, 'Stress Level (1-10)', 'Family History of Anxiety', 'Yes', 'No',alternative="two-sided")

U statistic = 539585.5000
p-value = 0.0530


Do individuals who smoke significantly differ in their average sleep duration compared to non-smokers?

In [17]:
compare(df, 'Sleep Hours', 'Smoking', 'Yes', 'No')

normality test: 0.0000,  0.0557
use mann_whitney
U statistic = 471270.0000
p-value = 0.0041


There is a statistically significant difference in sleep hours between smokers and non-smokers (p < 0.05). This suggests that smoking habits are associated with variations in the duration of sleep.

In [18]:
mann_whitney(df, 'Sleep Hours', 'Smoking', 'Yes', 'No',alternative="two-sided")

U statistic = 471270.0000
p-value = 0.0041


 Is there a statistically significant difference in heart rate (bpm) between individuals who take medication and those who do not?

In [19]:
compare(df, 'Heart Rate (bpm)', 'Medication', 'Yes', 'No')

normality test: 0.0000,  0.0000
use mann_whitney
U statistic = 477966.0000
p-value = 0.0071


There is a statistically significant difference in heart rate between those who take medication and those who do not (p < 0.05). This suggests that medication usage significantly correlates with heart rate patterns in the studied population.

In [20]:
mann_whitney(df, 'Heart Rate (bpm)', 'Medication', 'Yes', 'No', alternative="two-sided" )

U statistic = 477966.0000
p-value = 0.0071


Is there a statistically significant difference in weekly physical activity levels (hours/week) between individuals who have recently experienced a major life event and those who have not?

In [21]:
compare(df, 'Physical Activity (hrs/week)', 'Recent Major Life Event', 'Yes', 'No')

normality test: 0.0000,  0.0000
use mann_whitney
U statistic = 505417.5000
p-value = 0.4772


There is a statistically significant difference in physical activity levels between those who have experienced a major life event and those who have not (p < 0.05). This indicates that major life events significantly influence the amount of weekly physical activity.

In [22]:
mann_whitney(df, 'Physical Activity (hrs/week)', 'Recent Major Life Event', 'Yes', 'No', alternative='two-sided')

U statistic = 505417.5000
p-value = 0.4772


 Is there a statistically significant difference in anxiety levels (on a scale of 1-10) between individuals who experience dizziness and those who do not?

In [23]:
compare(df, 'Anxiety Level (1-10)', 'Dizziness', 'Yes', 'No')

normality test: 0.0000,  0.0000
use mann_whitney
U statistic = 545080.0000
p-value = 0.0189


 There is a statistically significant difference in anxiety levels between individuals who experience dizziness and those who do not (p < 0.05). This suggests that dizziness is associated with variations in anxiety levels.

In [24]:
mann_whitney(df, 'Anxiety Level (1-10)', 'Dizziness', 'Yes', 'No', alternative='two-sided')

U statistic = 545080.0000
p-value = 0.0189


Is there a statistically significant difference in diet quality between individuals in the therapy group and those in the no-therapy group?

In [25]:
compare(df, 'Diet Quality (1-10)', 'Therapy_Group', 'Therapy', 'No Therapy')

normality test: 0.0000,  0.0000
use mann_whitney
U statistic = 339880.0000
p-value = 0.0427


 Although the p-value (0.0427) is statistically significant (p < 0.05), it is relatively close to the threshold. This suggests that while there is a discernible difference in diet quality between the ‘Therapy’ and ‘No Therapy’ groups, the effect size may be moderate. It is recommended to interpret this finding as a potential indicator of a positive lifestyle shift associated with therapy, rather than a definitive causal link. Further longitudinal analysis would be beneficial to confirm if therapy directly influences dietary habits over time.

In [26]:
mann_whitney(df, 'Diet Quality (1-10)', 'Therapy_Group', 'Therapy', 'No Therapy', alternative='two-sided')

U statistic = 339880.0000
p-value = 0.0427


Is there a statistically significant association between gender and smoking habits?

No statistically significant association was found between gender and smoking habits. The variables are independent.

In [29]:
chi_2_test(df, 'Gender', 'Smoking')

Chi-2 Test
--------------------
Chi-2= 0.9487
p-value = 0.6223
df = 2
fail to reject h0(independent)


 Is there a statistically significant association between binned anxiety levels and therapy group membership?

There is a statistically significant association between anxiety levels and therapy groups. This suggests that therapy groups are not randomly distributed regarding anxiety levels, indicating a potential correlation

In [30]:
chi_2_test(df, 'Binned_Anxiety Level (1-10)', 'Therapy_Group')

Chi-2 Test
--------------------
Chi-2= 54.6900
p-value = 0.0000
df = 2
reject h0


 No statistically significant association exists between gender and therapy group membership. Gender does not influence which therapy group an individual is assigned to.

In [32]:
chi_2_test(df, 'Gender', 'Therapy_Group')

Chi-2 Test
--------------------
Chi-2= 0.1152
p-value = 0.9440
df = 2
fail to reject h0(independent)


: There is a strong, positive, and statistically significant correlation between stress levels and anxiety levels (p < 0.001), indicating that higher stress is strongly associated with higher anxiety.

In [33]:
correlation(df, 'Stress Level (1-10)', 'Anxiety Level (1-10)')

normality test: 0.0000,  0.0000
spearman
correlation : 0.6988
p_value: 0.0000


There is a statistically significant negative correlation between sleep hours and anxiety levels, indicating that increased sleep duration is associated with lower levels of anxiety.

In [34]:
correlation(df, 'Sleep Hours', 'Anxiety Level (1-10)')

normality test: 0.0000,  0.0000
spearman
correlation : -0.3434
p_value: 0.0000


 No statistically significant correlation was found between caffeine intake and heart rate (p > 0.05), suggesting that within this group, caffeine consumption does not significantly predict heart rate.

In [35]:
correlation(df, 'Caffeine Intake (mg/day)', 'Heart Rate (bpm)')

normality test: 0.0000,  0.0000
spearman
correlation : 0.0287
p_value: 0.1966


A statistically significant negative correlation exists between physical activity and stress levels, confirming that higher levels of activity correlate with reduced stress.

In [36]:
correlation(df, 'Physical Activity (hrs/week)', 'Stress Level (1-10)')

normality test: 0.0000,  0.0000
spearman
correlation : -0.1124
p_value: 0.0000


There is a statistically significant correlation between alcohol consumption and anxiety levels (p < 0.05), indicating that alcohol intake is associated with variations in reported anxiety.

In [37]:
correlation(df, 'Alcohol Consumption (drinks/week)', 'Anxiety Level (1-10)')

normality test: 0.0000,  0.0000
spearman
correlation : 0.0512
p_value: 0.0212


No statistically significant correlation was found between age and anxiety levels (p > 0.05), suggesting age is not a determining factor for anxiety in this group.

In [38]:
correlation(df, 'Age', 'Anxiety Level (1-10)')

normality test: 0.0000,  0.0000
spearman
correlation : -0.0316
p_value: 0.1542
